# Timed-Release Cooperative Roles

Train and inspect the 8-ant shared-writes cooperative checkpoint with fixed release-rank timing.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


/home/juan/reinforcement_learning/cool-antz/.venv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


{'jax_already_imported': False,
 'jax_preallocate': 'false',
 'jax_memory_fraction': '0.35',
 'jax_allocator': 'platform',
 'memory_trimmed': True,
 'disk_free_gb': 13.24,
 'disk_used_percent': 72.5,
 'current_pid': 5639,
 'safe_cleanup_candidate_count': 1581,
 'safe_cleanup_candidate_gb': 0.147,
 'top_memory_processes': [{'pid': 2713,
   'ppid': 1566,
   'rss_mb': 1254.8,
   'command': '/home/juan/reinforcement_learning/cool-antz/.venv/bin/python -m ipykernel_launcher --f=/run/user/1002/jupyter/runtime/kernel-v397af4bd50b79bcc0c1fab856003a74facdbd5d6a.json',
   'connection_file': '/run/user/1002/jupyter/runtime/kernel-v397af4bd50b79bcc0c1fab856003a74facdbd5d6a.json',
   'is_current_process': False,
   'is_notebook_kernel': True},
  {'pid': 1566,
   'ppid': 1317,
   'rss_mb': 796.8,
   'command': '/home/juan/.vscode-server/cli/servers/Stable-1b50d58d73426c9171299ec4037d01365d995b78/server/node --dns-result-order=ipv4first /home/juan/.vscode-server/cli/servers/Stable-1b50d58d73426c91712

In [ ]:
import importlib
import json
import pickle
import re
import shutil

import jax
from tqdm.auto import tqdm

from ant_byte_env.experiments import config_args_to_argv
from ant_byte_env.runs import write_json
from ant_byte_env.training.jax_mappo.timed_release import evaluation as timed_evaluation
from ant_byte_env.training.jax_mappo.timed_release import rendering as timed_rendering
from ant_byte_env.training.jax_mappo.timed_release import runner as timed_runner

workflows = importlib.reload(workflows)
timed_evaluation = importlib.reload(timed_evaluation)
timed_rendering = importlib.reload(timed_rendering)
timed_runner = importlib.reload(timed_runner)
print(f"JAX device: {jax.devices()[0]}")


/home/juan/reinforcement_learning/cool-antz/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX device: cuda:0


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "timed_release_roles_8ants_shared_writes.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_NAME = experiment.name
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
EVAL_DIR = RUN_DIR / "evaluation"
for directory in (CHECKPOINT_DIR, MEDIA_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Restore the source checkpoint first: {SOURCE_CHECKPOINT}")
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
BEST_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
TRAINING_LEDGER_PATH = RUN_DIR / "training_chunks.json"

UPDATE_TIMESTEPS = int(experiment_args["num_envs"]) * int(experiment_args["num_steps"])
BASE_TOTAL_UPDATES = int(experiment_args["total_timesteps"]) // UPDATE_TIMESTEPS
CHUNK_UPDATES = int(experiment.metadata.get("chunk_updates", 100))
BEST_EVAL_INTERVAL_UPDATES = int(experiment_args.get("best_eval_interval") or CHUNK_UPDATES)
CONTINUE_FROM_LATEST_CHUNK = bool(experiment.metadata.get("continue_from_latest_chunk", False))
CONTINUATION_UPDATES = int(experiment.metadata.get("continuation_updates", BASE_TOTAL_UPDATES))
EVALUATION_EPISODES = int(experiment.metadata.get("evaluation_episodes", 4))
RENDER_ACTION_MODE = str(experiment.metadata.get("render_action_mode", "sampled_move_greedy_write"))
RENDER_MAX_FRAMES = int(experiment.metadata.get("render_max_frames", 480))
RENDER_TILE_SIZE = int(experiment.metadata.get("render_tile_size", workflows.NOTEBOOK_ROLLOUT_TILE_SIZE))

RUN_TRAINING = True
MAX_CHUNKS_TO_RUN = None
RESUME_FROM_BEST_CHECKPOINT = False
RUN_BEST_EVAL_DURING_TRAINING = bool(experiment.metadata.get("run_best_eval_during_training", False))

existing_chunks = []
START_UPDATE = 0
NEXT_CHUNK_NUMBER = 1
latest_chunk_checkpoint = None

def chunk_label_parts(label):
    match = re.search(r"chunk_(\d+)_updates_(\d+)_(\d+)", str(label))
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    match = re.search(r"updates_(\d+)_(\d+)", str(label))
    if match:
        return 0, int(match.group(1)), int(match.group(2))
    return None

if TRAINING_LEDGER_PATH.exists():
    ledger_payload = json.loads(TRAINING_LEDGER_PATH.read_text())
    existing_chunks = list(ledger_payload.get("chunks", []))
    checkpoint_from_ledger = ledger_payload.get("terminal_checkpoint") or ledger_payload.get("active_checkpoint")
    if checkpoint_from_ledger:
        latest_chunk_checkpoint = Path(checkpoint_from_ledger)
        if not latest_chunk_checkpoint.is_absolute():
            latest_chunk_checkpoint = PROJECT_ROOT / latest_chunk_checkpoint
    for chunk in existing_chunks:
        parts = chunk_label_parts(chunk.get("label", ""))
        if parts is not None:
            chunk_number, _, chunk_end_update = parts
            START_UPDATE = max(START_UPDATE, chunk_end_update)
            NEXT_CHUNK_NUMBER = max(NEXT_CHUNK_NUMBER, chunk_number + 1)

disk_chunks = []
for checkpoint_path in CHECKPOINT_DIR.glob("chunk_*updates_*.pkl"):
    if checkpoint_path.name.endswith(".best_candidate.pkl"):
        continue
    parts = chunk_label_parts(checkpoint_path.stem)
    if parts is None:
        continue
    chunk_number, _, chunk_end_update = parts
    disk_chunks.append((chunk_end_update, chunk_number, checkpoint_path, checkpoint_path.stem))
if disk_chunks:
    disk_chunks.sort()
    latest_disk_end_update, latest_disk_chunk_number, latest_disk_checkpoint, _ = disk_chunks[-1]
    latest_chunk_checkpoint = latest_disk_checkpoint
    START_UPDATE = max(START_UPDATE, latest_disk_end_update)
    NEXT_CHUNK_NUMBER = max(NEXT_CHUNK_NUMBER, latest_disk_chunk_number + 1)
    existing_by_label = {str(chunk.get("label", "")): chunk for chunk in existing_chunks}
    for _, _, checkpoint_path, label in disk_chunks:
        existing_by_label.setdefault(
            label,
            {"label": label, "checkpoint": str(checkpoint_path), "recovered_from_disk": True},
        )
    existing_chunks = sorted(
        existing_by_label.values(),
        key=lambda chunk: (chunk_label_parts(chunk.get("label", "")) or (0, 0, 0))[2],
    )

if CONTINUE_FROM_LATEST_CHUNK and latest_chunk_checkpoint is not None and latest_chunk_checkpoint.exists():
    ACTIVE_CHECKPOINT = latest_chunk_checkpoint
    TRAINING_UPDATES = CONTINUATION_UPDATES
else:
    ACTIVE_CHECKPOINT = (
        BEST_CHECKPOINT_PATH
        if RESUME_FROM_BEST_CHECKPOINT and BEST_CHECKPOINT_PATH.exists()
        else SOURCE_CHECKPOINT
    )
    TRAINING_UPDATES = CONTINUATION_UPDATES
    START_UPDATE = 0
    existing_chunks = []
BASELINE_CHECKPOINT = ACTIVE_CHECKPOINT
TOTAL_UPDATES = TRAINING_UPDATES
PLANNED_FINAL_UPDATE = START_UPDATE + TRAINING_UPDATES
BEST_MODEL_METRIC = str(experiment_args.get("best_model_metric", "eval_mean_delivered_fraction"))
BEST_MODEL_MODE = str(experiment_args.get("best_model_mode", "max"))

def checkpoint_metric_value(path):
    if not path.exists():
        return None
    with path.open("rb") as checkpoint_file:
        checkpoint = pickle.load(checkpoint_file)
    value = checkpoint.get("metrics", {}).get(BEST_MODEL_METRIC)
    return None if value is None else float(value)

def metric_is_better(value, best_value):
    if best_value is None:
        return True
    if BEST_MODEL_MODE == "min":
        return value < best_value
    return value > best_value

BASELINE_BEST_METRIC_VALUE = checkpoint_metric_value(BASELINE_CHECKPOINT)
GLOBAL_BEST_METRIC_VALUE = checkpoint_metric_value(BEST_CHECKPOINT_PATH)
if GLOBAL_BEST_METRIC_VALUE is None:
    GLOBAL_BEST_METRIC_VALUE = BASELINE_BEST_METRIC_VALUE

summary = {
    "experiment": experiment.name,
    "source_checkpoint": str(SOURCE_CHECKPOINT),
    "baseline_checkpoint": str(BASELINE_CHECKPOINT),
    "run_dir": str(RUN_DIR),
    "base_total_updates": BASE_TOTAL_UPDATES,
    "training_updates_this_run": TRAINING_UPDATES,
    "start_update": START_UPDATE,
    "planned_final_update": PLANNED_FINAL_UPDATE,
    "chunk_updates": CHUNK_UPDATES,
    "best_eval_interval_updates": BEST_EVAL_INTERVAL_UPDATES,
    "existing_chunks": len(existing_chunks),
    "next_chunk_number": NEXT_CHUNK_NUMBER,
    "release_interval": experiment_args["release_interval"],
    "initial_active_ants": experiment_args["initial_active_ants"],
    "max_steps": experiment_args["max_steps"],
    "num_envs": experiment_args["num_envs"],
    "num_steps": experiment_args["num_steps"],
    "critic_architecture": experiment_args["critic_architecture"],
    "actor_only_warm_start": experiment_args.get("actor_only_warm_start", False),
    "active_checkpoint": str(ACTIVE_CHECKPOINT),
    "wandb_project": experiment_args.get("wandb_project"),
    "wandb_entity": experiment_args.get("wandb_entity"),
    "wandb_group": experiment_args.get("wandb_group"),
    "wandb_mode": experiment_args.get("wandb_mode"),
    "training_rollout_temperature": experiment_args["training_rollout_temperature"],
    "eval_move_temperature": experiment_args["best_eval_move_temperature"],
    "continue_from_latest_chunk": CONTINUE_FROM_LATEST_CHUNK,
    "run_best_eval_during_training": RUN_BEST_EVAL_DURING_TRAINING,
    "best_model_metric": BEST_MODEL_METRIC,
    "best_model_mode": BEST_MODEL_MODE,
    "baseline_best_metric_value": BASELINE_BEST_METRIC_VALUE,
    "current_global_best_metric_value": GLOBAL_BEST_METRIC_VALUE,
}
print(json.dumps(summary, indent=2))


{
  "experiment": "timed_release_roles_8ants_shared_writes_tuned",
  "source_checkpoint": "/home/juan/reinforcement_learning/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes/checkpoints/best_timed_release_roles_8ants_shared_writes.pkl",
  "baseline_checkpoint": "/home/juan/reinforcement_learning/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes/checkpoints/best_timed_release_roles_8ants_shared_writes.pkl",
  "run_dir": "/home/juan/reinforcement_learning/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes_tuned",
  "base_total_updates": 2000,
  "training_updates_this_run": 2000,
  "start_update": 0,
  "planned_final_update": 2000,
  "chunk_updates": 500,
  "best_eval_interval_updates": 500,
  "existing_chunks": 0,
  "next_chunk_number": 1,
  "release_interval": 150,
  "initial_active_ants": 1,
  "max_steps": 2000,
  "num_envs": 128,
  "num_steps": 256,
  "critic_architecture": "strided_cnn",
  "actor_only_warm_start": false,
  "active_checkpoi

In [ ]:
def progress_bar(label, total_updates):
    bar = tqdm(total=total_updates, desc=label)
    last = 0

    def callback(update, total, metrics):
        nonlocal last
        bar.total = total
        bar.update(max(0, int(update) - last))
        last = int(update)
        bar.set_postfix(
            ret=f"{metrics.get('episode_return', 0.0):.2f}",
            active=f"{metrics.get('mean_active_ants', 0.0):.2f}",
            delivered=f"{metrics.get('eval_mean_delivered_fraction', 0.0):.2f}",
        )

    return bar, callback

if RUN_TRAINING:
    previous_checkpoint = ACTIVE_CHECKPOINT
    chunk_count = (TRAINING_UPDATES + CHUNK_UPDATES - 1) // CHUNK_UPDATES
    if MAX_CHUNKS_TO_RUN is not None:
        chunk_count = min(chunk_count, int(MAX_CHUNKS_TO_RUN))
    chunk_metrics = []
    completed_updates = 0
    for chunk_index in range(chunk_count):
        updates = min(CHUNK_UPDATES, TRAINING_UPDATES - completed_updates)
        if updates <= 0:
            break
        chunk_start_update = START_UPDATE + completed_updates
        chunk_end_update = chunk_start_update + updates
        chunk_number = NEXT_CHUNK_NUMBER + chunk_index
        label = f"chunk_{chunk_number:03d}_updates_{chunk_start_update:05d}_{chunk_end_update:05d}"
        chunk_dir = RUN_DIR / label
        chunk_checkpoint = CHECKPOINT_DIR / f"{label}.pkl"
        chunk_best_checkpoint = CHECKPOINT_DIR / f"{label}.best_candidate.pkl"
        chunk_args = dict(experiment_args)
        chunk_args["total_timesteps"] = updates * UPDATE_TIMESTEPS
        chunk_args["load_model"] = str(previous_checkpoint)
        chunk_args["save_model"] = str(chunk_checkpoint)
        if chunk_args.get("wandb_run_name") is not None:
            chunk_args["wandb_run_name"] = f"{chunk_args['wandb_run_name']}_{label}"
        score_best_this_chunk = RUN_BEST_EVAL_DURING_TRAINING and (
            chunk_end_update == PLANNED_FINAL_UPDATE
            or chunk_end_update % BEST_EVAL_INTERVAL_UPDATES == 0
        )
        if score_best_this_chunk:
            chunk_args["save_best_model"] = str(chunk_best_checkpoint)
        else:
            chunk_args["save_best_model"] = None
            chunk_args["best_model_selection"] = "train"
        chunk_args["run_dir"] = str(chunk_dir)
        argv = config_args_to_argv(chunk_args)
        bar, callback = progress_bar(label, updates)
        try:
            metrics = timed_runner.main(argv, progress_callback=callback)
        finally:
            bar.close()
        chunk_entry = {"label": label, "checkpoint": str(chunk_checkpoint), "best_eval_scored": score_best_this_chunk, **metrics}
        if score_best_this_chunk and chunk_best_checkpoint.exists():
            candidate_metric_value = checkpoint_metric_value(chunk_best_checkpoint)
            promoted_to_best = candidate_metric_value is not None and metric_is_better(
                candidate_metric_value,
                GLOBAL_BEST_METRIC_VALUE,
            )
            chunk_entry.update(
                {
                    "candidate_best_checkpoint": str(chunk_best_checkpoint),
                    "candidate_best_metric_value": candidate_metric_value,
                    "promoted_to_best_checkpoint": promoted_to_best,
                }
            )
            if promoted_to_best:
                shutil.copy2(chunk_best_checkpoint, BEST_CHECKPOINT_PATH)
                GLOBAL_BEST_METRIC_VALUE = candidate_metric_value
                chunk_entry["global_best_checkpoint"] = str(BEST_CHECKPOINT_PATH)
        chunk_metrics.append(chunk_entry)
        previous_checkpoint = chunk_checkpoint
        completed_updates += updates
    terminal_checkpoint = previous_checkpoint
    ACTIVE_CHECKPOINT = (
        BEST_CHECKPOINT_PATH
        if RUN_BEST_EVAL_DURING_TRAINING and BEST_CHECKPOINT_PATH.exists()
        else BASELINE_CHECKPOINT
    )
    all_chunks = [*existing_chunks, *chunk_metrics]
    write_json(
        TRAINING_LEDGER_PATH,
        {
            "chunks": all_chunks,
            "active_checkpoint": str(ACTIVE_CHECKPOINT),
            "terminal_checkpoint": str(terminal_checkpoint),
            "best_checkpoint": str(BEST_CHECKPOINT_PATH) if BEST_CHECKPOINT_PATH.exists() else None,
            "global_best_metric_value": GLOBAL_BEST_METRIC_VALUE,
            "start_update": START_UPDATE,
            "completed_new_updates": completed_updates,
            "actual_final_update": START_UPDATE + completed_updates,
        },
    )
else:
    chunk_metrics = []

ACTIVE_CHECKPOINT


chunk_001_updates_00000_00500:   0%|          | 0/500 [00:00<?, ?it/s]wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/juan/.netrc.
wandb: Currently logged in as: jkaplan (jerefigueiredo-universidad-de-san-andr-s) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


2026-07-04 16:49:29.378474: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-07-04 16:50:12.571601: E external/xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bw-input.3 = (f32[8192,32,25,25]{3,2,1,0}, u8[0]{0}) custom-call(%add.1472, %bitcast.222), window={size=3x3 stride=2x2 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_name="jit(<lambda>)/jit(main)/while/body/while/body/conv_general_dilated" source_file="/home/juan/reinforcement_learning/cool-antz/src/ant_byte_env/training/jax_mappo/models.py" source_line=311}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"

active_agent_fraction,▁▂▅███▂▅███▂▅▇██▂▅▇██
applied_write_action_nonzero_rate,▁▂▅███▂▅███▂▅▇██▂▅▇██
approx_kl,▁▃█▄▄▄▂▂▂▄▄▃▂▂▂▄▁▄▄▂▄
best_eval_episodes,▁▁
best_eval_seed_offset,▁▁
best_model_metric_value,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
carrying_write_action_nonzero_rate,▁▂▅▆▆█▃▄▆▆▆▁▄▇▆▃▃▄▃▃▅
clipfrac,▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
completed_episodes,▁▁▁▁▁█▁▁▁▁█▁▁▁▁█▁▁▁▁█
delivery_events,▁▂▅█▆▅▂▆▇▆▆▂▆▇█▆▂▅█▆▅
+106,...


chunk_002_updates_00500_01000:   0%|          | 0/500 [00:00<?, ?it/s]

chunk_002_updates_00500_01000: 100%|██████████| 500/500 [10:37<00:00,  2.17s/it, active=7.55, delivered=0.33, ret=4.68]  

...

chunk_004_updates_01500_02000:   0%|          | 0/500 [00:00<?, ?it/s]

chunk_004_updates_01500_02000: 100%|██████████| 500/500 [10:39<00:00,  2.17s/it, active=7.60, delivered=0.35, ret=5.35]  

active_agent_fraction,▁▂▅███▂▅███▂▅▇██▃▅▇██
applied_write_action_nonzero_rate,▁▂▅███▂▅███▂▅███▃▅▇██
approx_kl,▃▂▂▂▃▇▂█▃▁▄▁▃▁▂▄▁▃▁▁▂
best_eval_episodes,▁▁
best_eval_seed_offset,▁▁
best_model_metric_value,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█
carrying_write_action_nonzero_rate,▁▃▆▅▄▇▄▅▅▅▆▅▅█▇▆▄▄▄▅▇
clipfrac,▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
completed_episodes,▁▁▁▁▁█▁▁▁▁█▁▁▁▁█▁▁▁▁▇
delivery_events,▁▂▄▇▆▅▂▄▆▇▆▂▆▇▆▅▂▅█▆▅
+106,...


chunk_004_updates_01500_02000: 100%|██████████| 500/500 [10:43<00:00,  1.29s/it, active=7.60, delivered=0.35, ret=5.35]


PosixPath('/home/juan/reinforcement_learning/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes_tuned/checkpoints/best_timed_release_roles_8ants_shared_writes_tuned.pkl')

In [ ]:
eval_metrics = timed_evaluation.evaluate_checkpoint(
    ACTIVE_CHECKPOINT,
    num_episodes=EVALUATION_EPISODES,
    action_mode=RENDER_ACTION_MODE,
    move_temperature=float(experiment_args.get("best_eval_move_temperature", 0.52)),
    write_temperature=float(experiment_args.get("best_eval_write_temperature", 1.0)),
)
eval_path = EVAL_DIR / f"timed_release_eval_{EVALUATION_EPISODES}ep.json"
write_json(eval_path, {"checkpoint": str(ACTIVE_CHECKPOINT), "metrics": eval_metrics})
eval_path, eval_metrics


(PosixPath('/home/juan/reinforcement_learning/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes_tuned/evaluation/timed_release_eval_16ep.json'),
 {'eval_success_rate': 0.0,
  'eval_mean_delivered_food': 53.8125,
  'eval_mean_delivered_fraction': 0.4305,
  'eval_mean_episode_return': 52.64918143106297,
  'eval_mean_episode_length': 2000.0,
  'eval_mean_active_ant_steps': 11800.0,
  'eval_mean_delivered_food_per_1000_active_ant_steps': 4.560380935668945,
  'eval_mean_pickups': 56.4375,
  'eval_rank_0_mean_pickups': 10.75,
  'eval_rank_0_mean_deliveries': 10.5625,
  'eval_rank_0_mean_writes': 1964.0,
  'eval_rank_0_mean_unique_cells': 828.5625,
  'eval_rank_0_mean_first_pickup_step': 511.875,
  'eval_rank_0_mean_first_delivery_step': 567.25,
  'eval_rank_0_mean_release_to_pickup_latency': 511.875,
  'eval_rank_1_mean_pickups': 10.125,
  'eval_rank_1_mean_deliveries': 9.6875,
  'eval_rank_1_mean_writes': 1821.9375,
  'eval_rank_1_mean_unique_cells': 815.4375,
  'eval_rank_1_

In [ ]:
video_path = timed_rendering.render_timed_release_checkpoint(
    ACTIVE_CHECKPOINT,
    MEDIA_DIR / "timed_release_roles_rollout.mp4",
    max_frames=RENDER_MAX_FRAMES,
    tile_size=RENDER_TILE_SIZE,
    action_mode=RENDER_ACTION_MODE,
    move_temperature=float(experiment_args.get("best_eval_move_temperature", 0.52)),
    write_temperature=float(experiment_args.get("best_eval_write_temperature", 1.0)),
)
video_path


/usr/lib/python3.10/subprocess.py:1796: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = _posixsubprocess.fork_exec(


PosixPath('/home/juan/reinforcement_learning/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes_tuned/media/timed_release_roles_rollout.mp4')